In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, RBF, Matern, WhiteKernel

from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.operators.sampling.lhs import LHS
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.indicators.hv import HV

In [ ]:
# start wall-clock timer for the whole notebook so total runtime is reported at the end
notebook_start = time.time()

## Global configuration of the GPR model

In [ ]:
# Problem constants 
CSV_FILE = "6 - MaintenancePlate_TrainingData.csv" # FEA dataset
INPUT_COLS = ["W1_m", "W2_m", "R_m", "t_m"] # 4 design variables 
STRESS_COL = "MaxVonMisesStress_MPa" # target: max von Mises stress from FEA
LOWER_BOUNDS = np.array([0.3, 0.1, 0.03, 0.01]) # design variable lower bounds
UPPER_BOUNDS = np.array([0.7, 0.15, 0.07, 0.02]) # design variable upper bounds
RHO = 2700.0 # aluminium alloy density 

# GPR defauts (they progressively get wiped out as the parameters are optimised)
KERNEL_DEFAULT = "RBF" # starting kernel
RESTARTS_DEFAULT = 5 # starting restarts 
SEED = 1 # Stays the same throughout optimisation 
THRESHOLD = 1.15 # Tolerance band used to pick fastest model within 15% of best RMSE

# Read the FEA dataset, trims it down to the first n rows 
def load_data(n_samples):
    df = pd.read_csv(CSV_FILE).iloc[:n_samples] # Return inputs and objective as NumPy arrays
    return df[INPUT_COLS].values, df[STRESS_COL].values

# Builds the kernel from the different defined types
def build_kernel(name, n_features):
    ls = np.ones(n_features) # initial length-scales (one per input)
    bounds = (1e-3, 1e3) # search range for variance and length-scales during fitting
    WK = WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-10, 1e0)) # noise term (default values for noise level and bounds)
    kernels = {"RBF": C(1, bounds) * RBF(ls, bounds),
        "Matern0.5": C(1, bounds) * Matern(ls, bounds, nu=0.5),
        "Matern1.5": C(1, bounds) * Matern(ls, bounds, nu=1.5),
        "Matern2.5": C(1, bounds) * Matern(ls, bounds, nu=2.5),
        "Matern2.5+WK": C(1, bounds) * Matern(ls, bounds, nu=2.5) + WK,
        "RBF+WK": C(1, bounds) * RBF(ls, bounds) + WK,}
    return kernels[name]

# Fits the GPR, predicts on the validation set, and returns the model and metrics
def train_and_evaluate_gpr(n_samples, kernel_name, n_restarts):
    X, y = load_data(n_samples) # Loads data
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.20, random_state=SEED, shuffle=True) # Splits it 80/20, fixed seed for reproducibility
    
    # Scale input and outputs 
    xs = MinMaxScaler() # Bounded design space maps to [0,1]
    ys = StandardScaler() # Standardise y
    X_tr_s = xs.fit_transform(X_tr) # fit + transform on training inputs
    X_val_s = xs.transform(X_val) # transform only on validation
    y_tr_s = ys.fit_transform(y_tr.reshape(-1, 1)).ravel() # reshape
    
    # Build kernel and GPR model 
    kernel = build_kernel(kernel_name, X_tr.shape[1]) # one of the kernels from build_kernel
    gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=n_restarts, random_state=SEED) # parameters
    t0 = time.time() # Calculates the training time only (fitting)
    gpr.fit(X_tr_s, y_tr_s)
    train_time = time.time() - t0

    # Predict on validation set and invert scaling so RMSE in MPa
    y_pred = ys.inverse_transform(gpr.predict(X_val_s).reshape(-1, 1)).ravel()
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2  = r2_score(y_val, y_pred)

    # Return everything downstream 
    return gpr, xs, ys, X_tr, X_val, y_tr, y_val, y_pred, rmse, r2, train_time


# Wraps the multi-objective problem (mass + stress) and runs NSGA-II
def run_nsga2(gpr, xs, ys, pop_size, n_gen):
    class PlateProblem(Problem):
        
        def __init__(self):
            super().__init__(n_var=4, n_obj=2, xl=LOWER_BOUNDS, xu=UPPER_BOUNDS) # 4 design variables w their bounds and 2 objectives
        
        def _evaluate(self, X, out, *args, **kwargs):
            W1, W2, R, t = X[:,0], X[:,1], X[:,2], X[:,3] # unpack the four design variables
            mass   = RHO * t * (4*W1**2 - 4*W2**2 + (4 - np.pi)*R**2) # analytical mass formula (Eq. 1)
            stress = ys.inverse_transform(gpr.predict(xs.transform(X)).reshape(-1,1)).ravel() # GPR prediction in MPa
            out["F"] = np.column_stack([mass, stress]) # both objectives to be minimised

    # NSGA-II setup: LHS for initial spread, SBX crossover and polynomial mutation are pymoo defaults
    alg = NSGA2(pop_size=pop_size, sampling=LHS(), crossover=SBX(prob=0.9, eta=15), mutation=PM(prob=0.2, eta=20), eliminate_duplicates=True)
    
    # Run optimisation, time it keep the history for plotting all evaluated points later
    t0  = time.time() # NSGA-II time
    res = minimize(PlateProblem(), alg, ("n_gen", n_gen), seed=SEED, verbose=False, save_history=True)
    return res, time.time() - t0

print("All functions defined")

## Optimisation 1 — Training Sample Size

In [ ]:
SAMPLE_SIZES = [50, 100, 150, 200, 250, 300]

rows = []
for n in SAMPLE_SIZES:
    # *_ ignores everything except the last three returns (Valuable metrics)
    *_, rmse, r2, t = train_and_evaluate_gpr(n, KERNEL_DEFAULT, RESTARTS_DEFAULT)
    # Store the run's metrics
    rows.append({"N_train": n, "RMSE (MPa)": round(rmse, 4), "R²": round(r2, 4), "Time (s)": round(t, 2)})
    # Prints one line per run so progress is visible while loop runs 
    print(f"N={n:>4}  /  RMSE: {rmse:.4f}  /  R²: {r2:.4f}  /  {t:.2f} s") # >4 left-pads for lineup

df_samples = pd.DataFrame(rows) # Convert the list of dicts into DataFrame
best_idx = df_samples["RMSE (MPa)"].idxmin() # Sample size with lowest RMSE
N_TRAIN_DEFAULT = df_samples.loc[best_idx, "N_train"] # Save it for next runs
print(f"\nSelected Sample Size = {N_TRAIN_DEFAULT}")

## Optimisation 2 — Kernel Type

In [ ]:
KERNEL_NAMES = ["RBF", "Matern0.5", "Matern1.5", "Matern2.5", "Matern2.5+WK", "RBF+WK",]

rows = []
for k in KERNEL_NAMES:
    *_, rmse, r2, t = train_and_evaluate_gpr(N_TRAIN_DEFAULT, k, RESTARTS_DEFAULT)
    rows.append({"Kernel": k, "RMSE (MPa)": round(rmse, 4), "R²": round(r2, 4), "Time (s)": round(t, 2)})
    print(f"Kernel: {k:>12}  /  RMSE: {rmse:.4f}  /  R²: {r2:.4f}  /  {t:.2f} s") # >12 left-pads for lineup

df_kernels = pd.DataFrame(rows)
best_rmse = df_kernels["RMSE (MPa)"].min()
candidates = df_kernels[df_kernels["RMSE (MPa)"] <= best_rmse * THRESHOLD] # Keeping kernels within 15% of threshold
KERNEL_DEFAULT = candidates.loc[candidates["Time (s)"].idxmin(), "Kernel"] # Of that list, choosign quickest one
print(f"\nSelected kernel = {KERNEL_DEFAULT} ")

There are possible warnings when launchint the kernel type selection. The ConstantKernel amplitude parameter can reach its upper bound of 1000, meaning the optimal value likely lies beyond this limit and the model may be slightly underfitted. Also, you can have the L-BFGS-B optimiser failing to converge within its iteration limit for the Matern2.5 + WK kernel, which indicates the loss landscape is complex enough that more iterations are needed to find a stable minimum. Both warnings come from the same underlying cause: the kernel hyperparameter search space and optimiser budget were not wide or deep enough to fully explore the solution for the more complex kernels, though the resulting models still achieve excellent accuracy.

## Optimisation 3 — Optimiser Restarts

In [ ]:
RESTARTS_VALUES = [0, 2, 3, 4, 5, 10]

rows = []
for r in RESTARTS_VALUES:
    *_, rmse, r2, t = train_and_evaluate_gpr(N_TRAIN_DEFAULT, KERNEL_DEFAULT, r)
    rows.append({"Restarts": r, "RMSE (MPa)": round(rmse, 4), "R²": round(r2, 4), "Time (s)": round(t, 2)})
    print(f"Restarts: {r:>2}  /  RMSE: {rmse:.4f}  /  R²: {r2:.4f}  /  {t:.2f} s")

df_restart = pd.DataFrame(rows)
best_rmse = df_restart["RMSE (MPa)"].min()
candidates = df_restart[df_restart["RMSE (MPa)"] <= best_rmse * THRESHOLD]
RESTARTS_DEFAULT = candidates.loc[candidates["Time (s)"].idxmin(), "Restarts"] # Of that list, choosign quickest one
print(f"\nSelected restart = {RESTARTS_DEFAULT} ")

Will have convergence error warnings for non-converged solutions when the # of restarts is not sufficient.

## Final GPR Configuration

In [ ]:
# Final GPR with the chosen settings
N_TRAIN_FINAL = N_TRAIN_DEFAULT # Optimised values set as final
KERNEL_FINAL = KERNEL_DEFAULT
RESTARTS_FINAL = RESTARTS_DEFAULT

# Train final GPR, keeping all returns for the plot below and for the NSGA-II step later
(gpr_final, xs_final, ys_final, X_tr_f, X_val_f, y_tr_f, y_val_f, y_pred_f,
rmse_f, r2_f, time_f) = train_and_evaluate_gpr(N_TRAIN_FINAL, KERNEL_FINAL, RESTARTS_FINAL)

# Prints the final solution, training points, and settings used
print(f"Final GPR -> RMSE: {rmse_f:.4f} MPa /  R²: {r2_f:.4f}  /  Time: {time_f:.2f} s")
print(f"Trained on {len(X_tr_f)} points, validated on {len(X_val_f)} points")
print(f"Settings ->  N={N_TRAIN_FINAL}  /  Kernel={KERNEL_FINAL}  /  Restarts={RESTARTS_FINAL}")

# Validation plot: predicted vs actual stress on the validation set
lims = [y_val_f.min(), y_val_f.max()] # axis limits for the ideal line
plt.figure(figsize=(6, 6))
plt.plot(lims, lims, "k--", label="Ideal") # y = x reference line
plt.scatter(y_val_f, y_pred_f, alpha=0.8) # one point per validation sample
plt.xlabel("Actual Validation Stress (MPa)")
plt.ylabel("Predicted Stress (MPa)")
plt.title("GPR Validation Accuracy")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## NSGA-II Hyperparameter Search

In [ ]:
COMBINATIONS = [( 50,  50), (100,  50), (100, 100), (150, 100), (150, 150)]

rows = []
for pop, gen in COMBINATIONS:
    res_i, t_i = run_nsga2(gpr_final, xs_final, ys_final, pop, gen)  # Run NSGA-II on trained GPR
    F_i = res_i.F # objective values of the Pareto front (mass, stress)
    n_pareto = len(F_i) # number of points on the front
    F_sorted = F_i[np.argsort(F_i[:, 0])] # Sort the front by mass so points are progressive neighbors 
    d = np.linalg.norm(np.diff(F_sorted, axis=0), axis=1) # Euclidean distance between each pair of consecutive Pareto points
     # Mean and std of spacing
    mean_sp = d.mean() if len(d) > 0 else 0.0 
    std_sp = d.std()  if len(d) > 0 else 0.0
    # Print results for each round
    rows.append({"Pop": pop, "Gen": gen, "Pareto pts": n_pareto, "Mean spacing": round(mean_sp, 4), "Spacing std": round(std_sp, 4), "Time (s)": round(t_i, 2)})
    print(f"Pop={pop:>4}   /  Gen={gen:>4}  /  Pareto: {n_pareto:>3}  /  Sp.mean: {mean_sp:.4f}  /  Sp.std: {std_sp:.4f}  |  {t_i:.2f} s")

# Choose lowest SP mean and SP STD
df_nsga = pd.DataFrame(rows)
df_nsga["Score"] = df_nsga["Mean spacing"] + df_nsga["Spacing std"]
best_idx = df_nsga["Score"].idxmin()
POP_FINAL = df_nsga.loc[best_idx, "Pop"]
GEN_FINAL = df_nsga.loc[best_idx, "Gen"]
print(f"\nSelected Population = {POP_FINAL} and Generations = {GEN_FINAL}")

## Final Run — GPR Training + Pareto Front

In [ ]:
# Getting training + Pareto runtime
# calls nsga-II function
res_final, time_nsga = run_nsga2(gpr_final, xs_final, ys_final, POP_FINAL, GEN_FINAL)

print(f"GPR training time : {time_f:.2f} s")
print(f"NSGA-II time      : {time_nsga:.2f} s")
print(f"Total pipeline    : {time_f + time_nsga:.2f} s")

## Pareto Front 

In [ ]:
# Pareto front analysis and plot
# Pull the design variables X and objective values F out of NSGA-II result
X_pareto = res_final.X 
F_pareto = res_final.F
mass_pareto = F_pareto[:, 0] 
stress_pareto = F_pareto[:, 1]

# Build  DataFrame so Pareto designs are easy to reference in the report
pareto_df = pd.DataFrame({"W1_m": X_pareto[:, 0], "W2_m": X_pareto[:, 1], "R_m":  X_pareto[:, 2], "t_m":  X_pareto[:, 3], "Mass_kg": mass_pareto, "PredictedStress_MPa": stress_pareto})

def normalise(s): return (s - s.min()) / (s.max() - s.min())
# Normalise both objectives to [0,1] so mass and stress can be compared
mass_n = normalise(pareto_df["Mass_kg"])
stress_n = normalise(pareto_df["PredictedStress_MPa"])
pareto_df["DistToIdeal"] = np.hypot(mass_n, stress_n) # Euclidean distance to (0,0)

# These are the solutions: 
lowest_stress = pareto_df["PredictedStress_MPa"].idxmin() # Lowest stress 
lowest_mass = pareto_df["Mass_kg"].idxmin() # Lowest mass 
optimal = pareto_df["DistToIdeal"].idxmin() # Optimal 

# Print the four design variables and both objectives for each notable design
cols = ["W1_m", "W2_m", "R_m", "t_m", "Mass_kg", "PredictedStress_MPa"]
print("Lowest stress point:");  print(pareto_df.loc[lowest_stress,  cols])
print("\nLowest mass point:");  print(pareto_df.loc[lowest_mass, cols])
print("\nOptimal compromise:"); print(pareto_df.loc[optimal, cols])

# Every solution NSGA-II ever evaluated across all generations
all_sols = np.vstack([g.pop.get("F") for g in res_final.history])
plt.figure(figsize=(7, 7))
plt.scatter(all_sols[:, 0], all_sols[:, 1], c="gray", alpha=0.25, label="All solutions") # All solutions (non-pareto)
plt.scatter(mass_pareto, stress_pareto, c="blue", label="Pareto front") # Pareto solutions
plt.scatter(pareto_df.loc[lowest_stress, "Mass_kg"], pareto_df.loc[lowest_stress, "PredictedStress_MPa"], color="red",    s=120, zorder=5, label="Lowest stress")
plt.scatter(pareto_df.loc[lowest_mass, "Mass_kg"], pareto_df.loc[lowest_mass, "PredictedStress_MPa"], color="green",  s=120, zorder=5, label="Lowest mass")
plt.scatter(pareto_df.loc[optimal, "Mass_kg"], pareto_df.loc[optimal, "PredictedStress_MPa"], color="orange", s=120, zorder=5, label="Optimal compromise")
plt.xlabel("Mass (kg)")
plt.ylabel("Stress (MPa)")
plt.title("Pareto Front — GPR + NSGA-II")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# End of clock
notebook_elapsed = time.time() - notebook_start
print(f"Total wall-clock time: {notebook_elapsed:.1f} s")